In [12]:
from app import load_index
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

model, index, chunks = load_index()

print(f"โหลด Knowledge Base สำเร็จ")
print(f"จำนวน chunks ทั้งหมด: {len(chunks)}")

โหลด Knowledge Base สำเร็จ
จำนวน chunks ทั้งหมด: 9


In [13]:
for i, chunk in enumerate(chunks):
    print(f"\n===== Chunk {i} =====")
    print(chunk)


===== Chunk 0 =====
# MilkLab° Knowledge Base

===== Chunk 1 =====
## เกี่ยวกับร้าน

===== Chunk 2 =====
MilkLab° เป็นร้านนมสดกลางคืน เปิดทุกวันยกเว้นจันทร์ เวลา 20:00 ถึง 01:00 น.
ตั้งอยู่หน้ามหาวิทยาลัย รับ delivery ผ่าน LINE OA

===== Chunk 3 =====
## เมนูหลัก

===== Chunk 4 =====
- นมหมีฮอกไกโด: 65 บาท (นมสดฮอกไกโด + วิปครีม) ขนาด 350 ml
- นมโกโก้บราวนี่: 70 บาท (นมสด + ผงโกโก้พรีเมียม + ก้อนบราวนี่) ขนาด 400 ml
- นมเสาวรส: 60 บาท (นมสด + น้ำเสาวรสสด) ขนาด 350 ml
- นมเย็นใส่วุ้นนม: 55 บาท (นมสด + วุ้นนม) ขนาด 400 ml

===== Chunk 5 =====
## Allergen

===== Chunk 6 =====
- ทุกเมนูมี lactose จาก milk
- บราวนี่มี gluten (wheat)
- วิปครีมมี dairy
- ลูกค้าแพ้ถั่ว: เมนูทั้งหมดปลอดถั่ว

===== Chunk 7 =====
## FAQ

===== Chunk 8 =====
**ส่งได้ไกลแค่ไหน**: รัศมี 5 กม. ค่าส่ง 30 บาท
**จองล่วงหน้าได้ไหม**: ได้ ผ่าน LINE OA สั่งก่อน 19:00
**กิน vegan ได้ไหม**: ไม่มีเมนู vegan ทุกเมนูใส่นมวัว
**ออเดอร์ขั้นต่ำ**: ไม่มี


In [14]:
eval_data = [
    {
        "question": "ร้านเปิดวันไหนและกี่โมง",
        "ground_truth": [2]
    },
    {
        "question": "ร้านตั้งอยู่ที่ไหน",
        "ground_truth": [2]
    },
    {
        "question": "ร้านมีบริการเดลิเวอรีไหม",
        "ground_truth": [2, 8]
    },
    {
        "question": "นมหมีฮอกไกโดราคาเท่าไร",
        "ground_truth": [4]
    },
    {
        "question": "นมโกโก้บราวนี่มีส่วนผสมอะไรบ้าง",
        "ground_truth": [4]
    },
    {
        "question": "นมเสาวรสขนาดกี่มิลลิลิตร",
        "ground_truth": [4]
    },
    {
        "question": "เมนูไหนราคา 55 บาท",
        "ground_truth": [4]
    },
    {
        "question": "ทุกเมนูมีสารก่อภูมิแพ้อะไร",
        "ground_truth": [6]
    },
    {
        "question": "มีเมนู vegan หรือไม่",
        "ground_truth": [8]
    },
    {
        "question": "ส่งได้ไกลแค่ไหนและค่าส่งเท่าไร",
        "ground_truth": [8]
    }
]

print(f"จำนวนคำถามประเมิน: {len(eval_data)}")

จำนวนคำถามประเมิน: 10


In [15]:
def retrieve_for_eval(question, model, index, chunks, k=3):
    query_embedding = model.encode(
        [question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "chunk_id": int(idx),
            "chunk": chunks[idx],
            "score": float(score)
        })

    return results

In [16]:
results = []

for item in eval_data:
    retrieved = retrieve_for_eval(
        item["question"],
        model,
        index,
        chunks,
        k=3
    )

    retrieved_ids = [r["chunk_id"] for r in retrieved]
    ground_truth = set(item["ground_truth"])
    relevant_retrieved = ground_truth.intersection(retrieved_ids)

    precision_at_3 = len(relevant_retrieved) / 3
    recall_at_3 = len(relevant_retrieved) / len(ground_truth)

    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "retrieved_ids": retrieved_ids,
        "precision@3": precision_at_3,
        "recall@3": recall_at_3,
        "top1_score": retrieved[0]["score"]
    })

results_df = pd.DataFrame(results)
results_df

,question,ground_truth,retrieved_ids,precision@3,recall@3,top1_score
0,ร้านเปิดวันไหนและกี่โมง,[2],"[2, 1, 8]",0.333333,1.0,0.461590
1,ร้านตั้งอยู่ที่ไหน,[2],"[1, 3, 2]",0.333333,1.0,0.609602
2,ร้านมีบริการเดลิเวอรีไหม,"[2, 8]","[1, 8, 3]",0.333333,0.5,0.611471
3,นมหมีฮอกไกโดราคาเท่าไร,[4],"[4, 8, 2]",0.333333,1.0,0.821973
4,นมโกโก้บราวนี่มีส่วนผสมอะไรบ้าง,[4],"[6, 4, 2]",0.333333,1.0,0.470814
5,นมเสาวรสขนาดกี่มิลลิลิตร,[4],"[4, 0, 6]",0.333333,1.0,0.751748
6,เมนูไหนราคา 55 บาท,[4],"[8, 3, 6]",0.000000,0.0,0.495427
7,ทุกเมนูมีสารก่อภูมิแพ้อะไร,[6],"[5, 6, 3]",0.333333,1.0,0.545622
8,มีเมนู vegan หรือไม่,[8],"[8, 6, 3]",0.333333,1.0,0.508016
9,ส่งได้ไกลแค่ไหนและค่าส่งเท่าไร,[8],"[8, 4, 2]",0.333333,1.0,0.503279


In [17]:
mean_precision = results_df["precision@3"].mean()
mean_recall = results_df["recall@3"].mean()

print(f"Mean Precision@3: {mean_precision:.3f}")
print(f"Mean Recall@3: {mean_recall:.3f}")

Mean Precision@3: 0.300
Mean Recall@3: 0.850


In [18]:
plt.figure(figsize=(8,5))
plt.hist(results_df["top1_score"], bins=5)
plt.title("Histogram of Top-1 Similarity Scores")
plt.xlabel("Similarity Score")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

## Reflection

The retrieval system achieved an average Precision@3 of **0.300** and Recall@3 of **0.850**.

The high Recall indicates that most relevant chunks were successfully retrieved within the top-3 results. However, Precision@3 is relatively low because only one or two retrieved chunks are usually relevant while the remaining retrieved chunks are unrelated.

One question, "เมนูไหนราคา 55 บาท", failed to retrieve the correct chunk in the top-3 results. This suggests that semantic retrieval may not perform well on numeric or exact-value queries. Improving chunk organization, increasing retrieval quality, or using reranking techniques could improve performance.

Overall, the retrieval system is effective for this small knowledge base and provides sufficient context for the LLM to generate accurate answers.